[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.1_ray_serve/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.1_ray_serve/lab.ipynb)

# 7.1 Lab: Ray Serve for LLM Inference[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.1_ray_serve/lab.ipynb)[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.1_ray_serve/lab.ipynb)This lab simulates Ray Serve's autoscaling behavior and multi-model routing to build intuition for production configuration.

In [ ]:
# Install dependencies via subprocess for clean environmentimport subprocessimport sys# Install numpy for numerical computation# Install matplotlib for visualizationsubprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib"])

In [ ]:
# Import core libraries for simulation and plottingimport numpy as npimport matplotlib.pyplot as plt# --- Autoscaling Simulation Parameters ---# Total duration of the traffic simulation in secondsSIMULATION_DURATION_S = 600# Time granularity: one tick per secondTIME_STEP_S = 1# Ray Serve config: target queue depth per replica# When depth exceeds this, autoscaler considers scaling upTARGET_ONGOING_REQUESTS = 3# Minimum replicas (never scale below this)MIN_REPLICAS = 1# Maximum replicas (cost safety cap)MAX_REPLICAS = 8# Seconds queue must exceed target before adding a replicaUPSCALE_DELAY_S = 30# Seconds queue must be low before removing a replicaDOWNSCALE_DELAY_S = 120# Average LLM request processing time in seconds# LLM requests are long (seconds) vs traditional ML (milliseconds)AVG_REQUEST_DURATION_S = 4.0

## Experiment 1: Autoscaling Under Traffic SpikesWe simulate incoming request traffic with a spike pattern and observe how Ray Serve's queue-depth autoscaler responds. The key insight: LLM requests are long-lived (seconds, not milliseconds), so queue depth is a better scaling signal than CPU utilization.

In [ ]:
def generate_traffic_pattern(duration_s, time_step_s):    """Generate a realistic traffic pattern with baseline + two spikes.        Returns time array and requests-per-second array.    """    # Calculate total number of simulation steps    n_steps = duration_s // time_step_s    # Create time array for plotting (x-axis values)    time_arr = np.arange(n_steps) * time_step_s    # Baseline arrival rate: 2 requests per second    base_rate = 2.0    # Initialize all timesteps with baseline traffic    traffic = np.full(n_steps, base_rate)    # First spike: 5x baseline from t=100s to t=200s    # Simulates a moderate burst (e.g., marketing campaign)    spike_1_start = 100 // time_step_s    spike_1_end = 200 // time_step_s    traffic[spike_1_start:spike_1_end] = base_rate * 5    # Second spike: 8x baseline from t=350s to t=420s    # Simulates a severe burst (e.g., viral content)    spike_2_start = 350 // time_step_s    spike_2_end = 420 // time_step_s    traffic[spike_2_start:spike_2_end] = base_rate * 8    # Return both arrays for downstream use    return time_arr, trafficdef simulate_autoscaling(traffic, target_depth, min_rep, max_rep, upscale_delay, downscale_delay, avg_duration):    """Simulate Ray Serve queue-depth autoscaling algorithm.        Models the core loop: measure depth -> decide -> scale.    Returns replica count and queue depth arrays over time.    """    # Total number of timesteps to simulate    n_steps = len(traffic)    # Array to track replica count at each timestep    replicas = np.zeros(n_steps, dtype=int)    # Array to track per-replica queue depth at each timestep    queue_depth = np.zeros(n_steps)    # Initialize with minimum replicas    current_replicas = min_rep    # Pending requests waiting to be processed    pending = 0.0    # Counter: how many ticks we've wanted to scale up    upscale_counter = 0    # Counter: how many ticks we've wanted to scale down    downscale_counter = 0    for t in range(n_steps):        # New requests arrive at this timestep        pending += traffic[t]        # Each replica processes at rate 1/avg_duration per second        # This models continuous batching throughput        processed = current_replicas * (1.0 / avg_duration)        # Subtract processed requests from queue (floor at 0)        pending = max(0, pending - processed)        # Compute queue depth per replica (the scaling signal)        depth_per_replica = pending / max(current_replicas, 1)        # Record metrics for this timestep        replicas[t] = current_replicas        queue_depth[t] = depth_per_replica        # --- Autoscaling decision logic ---        # If queue depth exceeds target: consider scaling up        if depth_per_replica > target_depth:            # Increment upscale counter (must persist for delay period)            upscale_counter += 1            # Reset downscale counter (can't scale both directions)            downscale_counter = 0            # Only scale up after sustained high depth            if upscale_counter >= upscale_delay:                # Add one replica (capped at max)                current_replicas = min(current_replicas + 1, max_rep)                # Reset counter after scaling action                upscale_counter = 0        # If queue depth is well below target: consider scaling down        elif depth_per_replica < target_depth * 0.5:            # Increment downscale counter            downscale_counter += 1            # Reset upscale counter            upscale_counter = 0            # Only scale down after sustained low depth (longer delay)            if downscale_counter >= downscale_delay:                # Remove one replica (floored at min)                current_replicas = max(current_replicas - 1, min_rep)                # Reset counter after scaling action                downscale_counter = 0        else:            # In normal range: reset both counters            upscale_counter = 0            downscale_counter = 0    # Return arrays for plotting    return replicas, queue_depth# Generate the traffic pattern with configured parameterstime_arr, traffic = generate_traffic_pattern(SIMULATION_DURATION_S, TIME_STEP_S)# Run the autoscaling simulationreplicas, queue_depth = simulate_autoscaling(    traffic, TARGET_ONGOING_REQUESTS, MIN_REPLICAS, MAX_REPLICAS,    UPSCALE_DELAY_S, DOWNSCALE_DELAY_S, AVG_REQUEST_DURATION_S)

In [ ]:
# Visualize: 3-panel chart showing traffic, replicas, and queue depthfig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)# Panel 1: Incoming request rate over timeaxes[0].fill_between(time_arr, traffic, alpha=0.3, color='#2563eb')axes[0].plot(time_arr, traffic, color='#2563eb', linewidth=1.5)axes[0].set_ylabel('Requests/sec')axes[0].set_title('Ray Serve Autoscaling Simulation')axes[0].legend(['Incoming Traffic'], loc='upper right')# Panel 2: Active replica count (step function)axes[1].step(time_arr, replicas, where='post', color='#166534', linewidth=2)# Reference lines for min and max boundsaxes[1].axhline(y=MIN_REPLICAS, color='gray', linestyle='--', alpha=0.5, label='Min replicas')axes[1].axhline(y=MAX_REPLICAS, color='red', linestyle='--', alpha=0.5, label='Max replicas')axes[1].set_ylabel('Replicas')axes[1].legend(loc='upper right')# Panel 3: Queue depth per replica vs target thresholdaxes[2].plot(time_arr, queue_depth, color='#991b1b', linewidth=1.5)# Target line: the scaling trigger thresholdaxes[2].axhline(y=TARGET_ONGOING_REQUESTS, color='orange', linestyle='--', label='Target depth')axes[2].set_ylabel('Queue Depth/Replica')axes[2].set_xlabel('Time (seconds)')axes[2].legend(loc='upper right')# Tight layout to avoid label overlapplt.tight_layout()# Save figure for offline reviewplt.savefig('autoscaling_simulation.png', dpi=150, bbox_inches='tight')plt.show()# Print key metrics from the simulationprint(f"Peak replicas reached: {replicas.max()}")print(f"Peak queue depth per replica: {queue_depth.max():.1f}")# Key insight: replicas lag behind spikes due to upscale_delayprint(f"Upscale delay effect: {UPSCALE_DELAY_S}s before first scale-up")

## Experiment 2: Multi-Model Routing Cost AnalysisCompare the cost of routing all requests to a large model vs. using a router that sends simple queries to a small model. This is the primary cost optimization lever in Ray Serve.

In [ ]:
# --- Multi-Model Routing Parameters ---# Total requests processed per hourREQUESTS_PER_HOUR = 10000# Fraction of requests that are "simple" (factual, short)# These can be served by a smaller, cheaper modelSIMPLE_FRACTION = 0.7# Cost per 1M tokens for large model (Llama 70B on A100)LARGE_MODEL_COST_PER_M_TOKENS = 2.50# Cost per 1M tokens for small model (Llama 8B on A10G)SMALL_MODEL_COST_PER_M_TOKENS = 0.30# Average tokens consumed per request (prompt + generation)AVG_TOKENS_PER_REQUEST = 500def compute_hourly_cost(requests, simple_frac, large_cost, small_cost, tokens_per_req):    """Compare cost: single large model vs routed multi-model.        Returns (single_model_cost, routed_cost) in dollars per hour.    """    # Total tokens processed in one hour    total_tokens = requests * tokens_per_req    # Single model approach: ALL tokens processed by large model    single_model_cost = (total_tokens / 1_000_000) * large_cost    # Routed approach: split tokens between models    # Simple queries go to the cheap small model    simple_tokens = total_tokens * simple_frac    # Complex queries still go to the expensive large model    complex_tokens = total_tokens * (1 - simple_frac)    # Total routed cost: sum of both model costs    routed_cost = (simple_tokens / 1_000_000) * small_cost + (complex_tokens / 1_000_000) * large_cost    # Return both for comparison    return single_model_cost, routed_cost# Sweep across different simple-request fractions# This shows how savings scale with traffic compositionfractions = np.linspace(0.1, 0.9, 9)# Accumulate costs for each fractionsingle_costs = []routed_costs = []for frac in fractions:    # Compute cost at this fraction    s, r = compute_hourly_cost(        REQUESTS_PER_HOUR, frac,        LARGE_MODEL_COST_PER_M_TOKENS, SMALL_MODEL_COST_PER_M_TOKENS,        AVG_TOKENS_PER_REQUEST    )    # Store for plotting    single_costs.append(s)    routed_costs.append(r)# Plot cost comparison across routing fractionsfig, ax = plt.subplots(figsize=(10, 5))# Single model cost: flat (doesn't depend on routing fraction)ax.plot(fractions * 100, single_costs, 'o-', color='#991b1b', linewidth=2, label='Single Large Model')# Routed cost: decreases as more traffic goes to small modelax.plot(fractions * 100, routed_costs, 's-', color='#166534', linewidth=2, label='Routed (Large + Small)')# Shade the savings region between the two linesax.fill_between(fractions * 100, routed_costs, single_costs, alpha=0.15, color='#166534')# Axis labels and formattingax.set_xlabel('% Simple Requests Routed to Small Model')ax.set_ylabel('Hourly Cost ($)')ax.set_title('Multi-Model Routing: Cost Savings vs Single Model')ax.legend()ax.grid(True, alpha=0.3)plt.tight_layout()# Save figure for referenceplt.savefig('routing_cost_comparison.png', dpi=150, bbox_inches='tight')plt.show()# Print savings at the configured default fractions, r = compute_hourly_cost(    REQUESTS_PER_HOUR, SIMPLE_FRACTION,    LARGE_MODEL_COST_PER_M_TOKENS, SMALL_MODEL_COST_PER_M_TOKENS,    AVG_TOKENS_PER_REQUEST)# Calculate percentage savingssavings_pct = (1 - r/s) * 100print(f"At {SIMPLE_FRACTION*100:.0f}% simple routing: ${s:.2f}/hr -> ${r:.2f}/hr ({savings_pct:.0f}% savings)")